In [17]:
__author__ = "Sean Dang"
__PSID__ = "2518173"

In [18]:
# Data Manipulation and Visualization
import pandas as pd #creating and manipulating dataframes
import matplotlib.pyplot as plt #visuals
import seaborn as sns #visuals
from sklearn.cluster import KMeans #K-Means
from sklearn.cluster import DBSCAN #DBSCAN
import numpy as np
from sklearn.metrics import confusion_matrix

In [19]:
# Reading the data
data = pd.read_csv('clinical_records_dataset.csv')
class_labels = data['DEATH_EVENT']
data = data.drop('DEATH_EVENT', axis=1)
data = data.drop('time', axis=1)

# Task 1

In [20]:
''' Modify this method '''
def purity(y_true,y_pred):

    """
    Compute the purity score for clustering.

    Parameters:
    y_true (array-like): True class labels.
    y_pred (array-like): Predicted cluster labels.

    Returns:
    float: Purity score, a single number
    """
    contingency_matrix = confusion_matrix(y_true, y_pred)

    # Using amax to compute the number of data points assigned to the correct label, based on the contingency matrix
    correct = np.sum(np.amax(contingency_matrix, axis=0))

    # Find the total number of data points
    ndata = np.sum(contingency_matrix)
      

    return correct/ndata

# Task 2

In [21]:
#Run K-means on the dataset with k=2. Use the default parameters for the algorithm. 
#Compute the purity of the clustering result. 
#Compute the purity of the clustering result for each of the two clusters.

# Initialize and fit K-Means

kmeans = KMeans(n_clusters=2, random_state=42)
y_pred = kmeans.fit_predict(data)

# compute overall purity

overall_purity = purity(class_labels, y_pred)
print(f"Overall Purity: {overall_purity:.4f}")

results = pd.DataFrame({'True_Label': class_labels, 'Cluster_Label': y_pred})

for cluster_id in range(2):
    # Filter data for only this cluster
    cluster_subset = results[results['Cluster_Label'] == cluster_id]
    
    # Calculate purity for this specific cluster
    if len(cluster_subset) > 0:
        counts = cluster_subset['True_Label'].value_counts()
        cluster_purity = counts.max() / len(cluster_subset)
        percentage_of_total = (len(cluster_subset) / len(data)) * 100
        
        print(f"\nCluster {cluster_id}:")
        print(f" - Purity: {cluster_purity:.4f}")
        print(f" - Percentage of total data: {percentage_of_total:.2f}%")

Overall Purity: 0.6789

Cluster 0:
 - Purity: 0.6923
 - Percentage of total data: 78.26%

Cluster 1:
 - Purity: 0.6308
 - Percentage of total data: 21.74%


Which cluster has the highest purity? What percentage of the data points were assigned to this cluster? What percentage of the data points were assigned to the other cluster?

- Cluster 0 has the highest purity.
- 78.26% of the data points were assigned to this cluster (cluster 0).
- 21.74% of the data points were assigned to this cluster 1.

# Task 3


In [22]:
#Run K-means on the dataset with k=3 and k = 5. 
#Compute the overall purity of clustering and the purity of each cluster for each value of k.

k_values = [3,5]

for k in k_values:
    # Initialize and fit K-Means
    kmeans = KMeans(n_clusters=k, random_state=42)
    y_pred = kmeans.fit_predict(data)

    # Compute overall purity
    overall_purity = purity(class_labels, y_pred)
    print(f"Overall Purity: {overall_purity:.4f}\n")

    # Compute individual clusters
    results = pd.DataFrame({'True_Label': class_labels, 'Cluster_Label': y_pred})

    for cluster_id in range(k):
        # Filter data for only this cluster
        cluster_subset = results[results['Cluster_Label'] == cluster_id]
        
        # Calculate purity and percentage for this specific cluster
        if len(cluster_subset) > 0:
            counts = cluster_subset['True_Label'].value_counts()
            cluster_purity = counts.max() / len(cluster_subset)
            percentage_of_total = (len(cluster_subset) / len(data)) * 100
            
            print(f"Cluster {cluster_id}:")
            print(f" - Purity: {cluster_purity:.4f}")
            print(f" - Percentage of total data: {percentage_of_total:.2f}%\n")

Overall Purity: 0.6789

Cluster 0:
 - Purity: 0.7158
 - Percentage of total data: 61.20%

Cluster 1:
 - Purity: 0.6585
 - Percentage of total data: 13.71%

Cluster 2:
 - Purity: 0.6000
 - Percentage of total data: 25.08%

Overall Purity: 0.6789

Cluster 0:
 - Purity: 0.6993
 - Percentage of total data: 51.17%

Cluster 1:
 - Purity: 0.6000
 - Percentage of total data: 5.02%

Cluster 2:
 - Purity: 0.6271
 - Percentage of total data: 19.73%

Cluster 3:
 - Purity: 0.6857
 - Percentage of total data: 23.41%

Cluster 4:
 - Purity: 1.0000
 - Percentage of total data: 0.67%



Which value of k gives the best clustering result? Explain why.

Both `k=3` and `k=5` have 0.6789 overall purity so neither of them improves the clustering result. Furthermore, `k=5` begins to overfit to outliers. When we look at cluster 4 of `k=5`, we will see it contains only 0.67% which is 2 patients of the data. Because the data is unscaled, features with massive magnitudes like platelets are dominating the distance calculation, preventing K-means from finding meaningful clinical clusters. Therefore, `k=3` should be better option.

# Task 4

In [23]:
#Run DBSCAN on the dataset with minPts=5 and eps=0.5. 
#Compute the purity of the clustering result. 
#Compute the purity of the clustering result for each of the two clusters.

# Initialize and fit DBSCAN
# scikit-learn uses 'min_samples' instead of 'minPts'
dbscan = DBSCAN(eps=0.5, min_samples=5)
y_pred_db = dbscan.fit_predict(data)

# Compute overall purity
overall_purity_db = purity(class_labels, y_pred_db)

print(f"Overall Purity: {overall_purity_db:.4f}\n")

# Compute individual clusters
results_db = pd.DataFrame({'True_Label': class_labels, 'Cluster_Label': y_pred_db})

# Find all unique clusters DBSCAN created (including -1 for noise)
unique_clusters = np.unique(y_pred_db)

for cluster_id in unique_clusters:
    # Filter data for only this cluster
    cluster_subset = results_db[results_db['Cluster_Label'] == cluster_id]
    
    # Calculate purity and percentage for this specific cluster
    if len(cluster_subset) > 0:
        counts = cluster_subset['True_Label'].value_counts()
        cluster_purity = counts.max() / len(cluster_subset)
        percentage_of_total = (len(cluster_subset) / len(data)) * 100
        
        cluster_name = f"Cluster {cluster_id}" if cluster_id != -1 else "Noise (Cluster -1)"
        
        print(f"{cluster_name}:")
        print(f" - Purity: {cluster_purity:.4f}")
        print(f" - Percentage of total data: {percentage_of_total:.2f}%\n")

Overall Purity: 0.6789

Noise (Cluster -1):
 - Purity: 0.6789
 - Percentage of total data: 100.00%



Which cluster has the highest purity? What percentage of the data points were assigned to this cluster? What percentage of the data points were assigned to the other cluster?

- Cluster -1 (the Noise cluster) is the only cluster generated, so it technically has the highest purity which is 0.6789. Because the dataset is unscaled, the `eps` parameter = 0.5 is way too small to capture points that are tens of thousands of units apart like platelets. This causes DBSCAN to classify every single patient as an outlier/noise.
- 100% of the data points were assigned to cluster -1.
- There is no other cluster. DBSCAN failed to form any actual clusters because the `minPts` requirement of 5 could not be met within a extremely tiny radius of `eps=0.5`.

# Task 5


In [24]:
# Develop a search procedure to find the best parameters for DBSCAN. The parameters to search over are minPts and eps. 
# The procedure should maximize the purity of the clustering result, subject to the following constraints:
# 1. There should be between 2 and 18 clusters.
# 2. The percentage of outliers should be less than 10%.

# I think Grid Search is the best way to approach this problem since we only have 2 parameters.


eps_values = np.linspace(500, 10000, 20) # Testing 20 values from 500 to 10,000
minPts_values = range(2, 11) # Testing minPts from 2 to 10

best_purity = -1
best_params = {'eps': None, 'min_samples': None}
best_labels = None

print("Starting Search...")

for e in eps_values:
    for m in minPts_values:
        db = DBSCAN(eps=e, min_samples=m)
        labels = db.fit_predict(data)
        
        # Calculate number of clusters
        unique_labels = set(labels)
        n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
        
        # Calculate percentage of outliers
        n_noise = list(labels).count(-1)
        outlier_percent = (n_noise / len(data)) * 100
        
        # Check constraints: 2-18 clusters and < 10% outliers
        if 2 <= n_clusters <= 18 and outlier_percent < 10:
            current_purity = purity(class_labels, labels)
            
            if current_purity > best_purity:
                best_purity = current_purity
                best_params['eps'] = e
                best_params['min_samples'] = m
                best_labels = labels


if best_labels is not None:
    print(f"\nBest Parameters: eps={best_params['eps']}, min_samples={best_params['min_samples']}")
    print(f"Highest Overall Purity: {best_purity:.4f}")
    
    # Detailing the best result
    results_df = pd.DataFrame({'True': class_labels, 'Cluster': best_labels})
    for cid in sorted(set(best_labels)):
        subset = results_df[results_df['Cluster'] == cid]
        p = subset['True'].value_counts().max() / len(subset)
        label_name = f"Cluster {cid}" if cid != -1 else "Noise"
        print(f"{label_name} Purity: {p:.4f} (Size: {len(subset)})")
else:
    print("No combination satisfied the constraints. Try increasing the eps range.")

Starting Search...

Best Parameters: eps=8500.0, min_samples=2
Highest Overall Purity: 0.6890
Noise Purity: 0.7778 (Size: 9)
Cluster 0 Purity: 0.6939 (Size: 245)
Cluster 1 Purity: 0.6000 (Size: 5)
Cluster 2 Purity: 0.6000 (Size: 25)
Cluster 3 Purity: 0.6667 (Size: 3)
Cluster 4 Purity: 0.5000 (Size: 2)
Cluster 5 Purity: 0.6667 (Size: 3)
Cluster 6 Purity: 1.0000 (Size: 3)
Cluster 7 Purity: 0.7500 (Size: 4)


Which parameters give the best clustering result? What is the purity of the clustering result? What is the purity of the clustering result for each of the clusters? Which cluster has the highest purity?

- The parameter give the best clustering result is `eps = 8500` and `minPts = 2`.
- There Overall Purity is 0.6890.
- The purity results for each cluster are `0.6939`, `0.6000`, `0.6000`, `0.6667`, `0.5000`, `0.6667`, `1.0000`, `0.7500` respectively.
- Cluster 6 has the highest purity which is 1.0000. However, this cluster only has 3 data points, making it a very tiny cluster.